# 07c · HFC: clases, factores, representantes y bootstrap (vigente: niveles, 16 series)

**Objetivo.** Apunte 12 y OE5: describir y poner a prueba la partición de las series que da la
Clasificación Factorial Jerárquica del asesor (`clahfac`, port de Python `src/camiz_port.py`, D6),
con los mismos pasos que la sección 5 del Rmd final.

1. **Particiones en 3 y 4 clases**, deshaciendo las últimas fusiones (`partclas`, la misma
   partición de `clahpart`). Las clases se nombran **por composición** (C1, C2, … en el orden de la
   primera serie de cada clase), nunca por el número de nodo.
2. **Factor** de cada clase (factor A del nodo, `repvar`; la serie estandarizada si la clase tiene
   una sola serie) y **serie representante** (la serie de la clase más correlacionada, en valor
   absoluto, con el factor; definición operativa declarada, que se lleva como pregunta al
   Dr. Camiz). Para 3 clases: correlaciones entre factores y entre representantes, con intervalos
   p5–p95 por bootstrap de bloques de 24 meses de las filas (series fijas); en niveles, también sin
   la tendencia lineal de cada serie.
3. **Bootstrap de la HFC**: en cada réplica se recalcula la jerarquía y se miden el Rand ajustado
   de las particiones en 3 y 4 clases frente a las originales, la correlación cofenética con la
   jerarquía original y la frecuencia de coclasificación de cada par de series.

**Réplicas.** Las 500 réplicas de `R_bootstrap_indices_L24.csv` que sortea el Rmd (semilla 2026);
este notebook no sortea nada (`SEMILLA = None`). *Supuesto:* el bootstrap por bloques supone
dependencia débil; en niveles no se cumple, y las cifras miden estabilidad descriptiva.

**Sustituciones.** Ninguna: la HFC es la del asesor, no `hclust`/`scipy`.

**Parámetros** (`src/parametros.py`): por defecto, niveles, 16 series (D16), salidas en
`data/results/final/`. Con `TESIS_VIA=ambas` y `TESIS_EXCLUIDAS=[]`, el antecedente.

**Salidas:** `07c_hfc_clases3.csv`, `07c_hfc_correlaciones3.csv`, `07c_hfc_bootstrap.csv` y
`07c_hfc_coclasificacion.csv`. Sus homólogos de R son los `R_hfc_*` del mismo nombre.

In [1]:
# ---- Configuración y parámetros (adenda 3b, §3) ------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src.parametros import leer_parametros
from src.datos import cargar_vias, meta_vigente

PARAM = leer_parametros(RAIZ)
VIA, EXCLUIDAS, DIR, DIR_R = PARAM['VIA'], PARAM['EXCLUIDAS'], PARAM['DIR'], PARAM['DIR_R']
ANTECEDENTE = len(EXCLUIDAS) == 0          # 17 series: modo de no regresión
X, Xt = cargar_vias(RAIZ, excluidas=EXCLUIDAS)
META = meta_vigente(EXCLUIDAS)
P = X.shape[1]
TODAS = {'niveles': X, 'transformadas': Xt}
VIAS = {v: (TODAS[v], DIR[v], DIR_R[v]) for v in PARAM['VIAS_ACTIVAS']}
print('VIA =', VIA, '| EXCLUIDAS =', EXCLUIDAS, '| p =', P, '| CARPETA =', PARAM['CARPETA'])
for via, (M, d, dr) in VIAS.items():
    print(via, M.shape, M.index[0], '→', M.index[-1], '| salidas:', d.relative_to(RAIZ) if d.is_relative_to(RAIZ) else d)

from src import camiz_port as cp
from src.particion import estandarizar_r
from src.remuestreo import leer_inicios_bootstrap, indices_desde_inicios
from src.series_tiempo import quitar_tendencia

SEMILLA = None
L_BOOT = 24

VIA = niveles | EXCLUIDAS = ('Liquidez M2',) | p = 16 | CARPETA = /home/claude/tesis-modelamiento-matematico/data/results/final
niveles (228, 16) 2007-01 → 2025-12 | salidas: data/results/final


## 1. Clases por composición, factores y representantes

In [2]:
def clases_hfc(hier, ncl):
    pos = cp.partclas(hier, ncl)['partition']['Position'].to_numpy()
    u = list(dict.fromkeys(pos))                 # orden de aparición en el orden de las series
    return [{'clase': f'C{i + 1}', 'miembros': np.nonzero(pos == q)[0],
             'nodo': int(P + q) if q > 0 else int(-q)} for i, q in enumerate(u)]


def factor_clase(h, Z, cl):
    if cl['nodo'] <= P:
        return Z[:, cl['nodo'] - 1]
    return h['repvar'].iloc[:, 2 * (cl['nodo'] - P) - 2].to_numpy()


def composicion_hfc(h, Z):
    filas = []
    for ncl in (3, 4):
        for cl in clases_hfc(h['hier'], ncl):
            f = factor_clase(h, Z, cl)
            r = np.array([cp._cor_r(Z[:, j], f)[0, 0] for j in cl['miembros']])
            rep = int(np.argmax(np.abs(r)))
            for i, j in enumerate(cl['miembros']):
                filas.append({'particion': ncl, 'clase': cl['clase'], 'nodo': cl['nodo'],
                              'serie': META.nombre_corto[j], 'codigo': META.codigo[j],
                              'r_factor': r[i], 'representante': i == rep})
    return pd.DataFrame(filas)


hfc, comp = {}, {}
for via, (M, d, dr) in VIAS.items():
    h = cp.clahfac(M)
    Z = estandarizar_r(M.to_numpy())
    hfc[via] = (h, Z)
    comp[via] = composicion_hfc(h, Z)
    comp[via].to_csv(d / '07c_hfc_clases3.csv', index=False)
    print(via)
    display(comp[via].groupby(['particion', 'clase']).agg(series=('serie', ', '.join),
            representante=('serie', lambda s: s[comp[via].loc[s.index, 'representante']].iloc[0])))

niveles


series  \
particion clase                                                      
3         C1     Crédito empresarial, Crédito consumo, Crédito ...   
          C2         Tasa de referencia, Tasa interbancaria, TIPMN   
          C3                                Índice BVL, EMBIG Perú   
4         C1     Crédito empresarial, Crédito consumo, Crédito ...   
          C2         Tasa de referencia, Tasa interbancaria, TIPMN   
          C3                                            Índice BVL   
          C4                                            EMBIG Perú   

                       representante  
particion clase                       
3         C1     Crédito hipotecario  
          C2      Tasa interbancaria  
          C3              EMBIG Perú  
4         C1     Crédito hipotecario  
          C2      Tasa interbancaria  
          C3              Índice BVL  
          C4              EMBIG Perú

## 2. Correlaciones entre los factores y entre los representantes (3 clases)

In [3]:
PARES3 = [(0, 1), (0, 2), (1, 2)]

def cor_boot(F, inicios, largo):
    n = len(F)
    rb = np.array([[cp._cor_r(F[indices_desde_inicios(ini, largo, n)][:, a], F[indices_desde_inicios(ini, largo, n)][:, b])[0, 0]
                    for a, b in PARES3] for ini in inicios])
    return pd.DataFrame({'clase_a': [f'C{a + 1}' for a, _ in PARES3], 'clase_b': [f'C{b + 1}' for _, b in PARES3],
                         'r': [cp._cor_r(F[:, a], F[:, b])[0, 0] for a, b in PARES3],
                         'p5': np.quantile(rb, 0.05, axis=0), 'p95': np.quantile(rb, 0.95, axis=0),
                         'replicas': len(inicios)})


inicios = {via: leer_inicios_bootstrap(dr / 'R_bootstrap_indices_L24.csv') for via, (_, d, dr) in VIAS.items()}
corr = {}
for via, (M, d, dr) in VIAS.items():
    sin_tend = via == 'niveles'
    h, Z = hfc[via]
    cl3 = clases_hfc(h['hier'], 3)
    F = np.column_stack([factor_clase(h, Z, cl) for cl in cl3])
    c = comp[via]
    rep = [list(META.nombre_corto).index(s) for s in c.serie[(c.particion == 3) & c.representante]]
    R = Z[:, rep]
    partes = [('original', 'factores', F), ('original', 'representantes', R)]
    if sin_tend:
        partes += [('sin_tendencia', 'factores', quitar_tendencia(F)), ('sin_tendencia', 'representantes', quitar_tendencia(R))]
    corr[via] = pd.concat([cor_boot(Fm, inicios[via], L_BOOT).assign(version=v, tipo=t) for v, t, Fm in partes],
                          ignore_index=True)[['version', 'tipo', 'clase_a', 'clase_b', 'r', 'p5', 'p95', 'replicas']]
    corr[via].to_csv(d / '07c_hfc_correlaciones3.csv', index=False)
    print(via)
    display(corr[via])

niveles


,version,tipo,clase_a,clase_b,r,p5,p95,replicas
0,original,factores,C1,C2,-0.071211,-0.599696,0.438684,500
1,original,factores,C1,C3,-0.482760,-0.643966,-0.175732,500
2,original,factores,C2,C3,0.113633,-0.217825,0.470978,500
3,original,representantes,C1,C2,0.076276,-0.547494,0.535947,500
4,original,representantes,C1,C3,-0.263185,-0.473469,0.058846,500
5,original,representantes,C2,C3,0.262665,0.103902,0.459102,500
6,sin_tendencia,factores,C1,C2,-0.411427,-0.636524,-0.005231,500
7,sin_tendencia,factores,C1,C3,0.151590,-0.149477,0.401175,500
8,sin_tendencia,factores,C2,C3,0.114447,-0.092701,0.404415,500
9,sin_tendencia,representantes,C1,C2,0.358983,0.054417,0.540035,500


## 3. Bootstrap de la HFC

In [4]:
boot = {}
for via, (M, d, dr) in VIAS.items():
    h, _ = hfc[via]
    pos0 = {k: cp.partclas(h['hier'], k)['partition']['Position'].to_numpy() for k in (3, 4)}
    junto = {k: np.zeros((P, P)) for k in (3, 4)}
    filas = []
    A = M.to_numpy()
    for b, ini in enumerate(inicios[via], start=1):
        idx = indices_desde_inicios(ini, L_BOOT, len(A))
        hb = cp.clahfac(pd.DataFrame(A[idx], columns=M.columns))
        ari = {}
        for k in (3, 4):
            pb = cp.partclas(hb['hier'], k)['partition']['Position'].to_numpy()
            junto[k] += pb[:, None] == pb[None, :]
            ari[k] = cp.rand_ajustado(pb, pos0[k])
        filas.append({'replica': b, 'ari_3': ari[3], 'ari_4': ari[4], 'cofenetica': cp.cor_cofenetica(hb['hier'], h['hier'])})
    B = len(inicios[via])
    iu = [(i, j) for i in range(P) for j in range(i + 1, P)]
    co = pd.DataFrame({'serie_a': [META.nombre_corto[i] for i, _ in iu], 'serie_b': [META.nombre_corto[j] for _, j in iu],
                       'codigo_a': [META.codigo[i] for i, _ in iu], 'codigo_b': [META.codigo[j] for _, j in iu],
                       'frec_3': [junto[3][i, j] / B for i, j in iu], 'frec_4': [junto[4][i, j] / B for i, j in iu]})
    boot[via] = (pd.DataFrame(filas), co)
    boot[via][0].to_csv(d / '07c_hfc_bootstrap.csv', index=False)
    co.to_csv(d / '07c_hfc_coclasificacion.csv', index=False)
    print(via, B, 'réplicas · Rand ajustado mediano (3 / 4 clases):',
          round(float(boot[via][0].ari_3.median()), 3), '/', round(float(boot[via][0].ari_4.median()), 3),
          '· cofenética mediana:', round(float(boot[via][0].cofenetica.median()), 3))

niveles 500 réplicas · Rand ajustado mediano (3 / 4 clases): 1.0 / 1.0 · cofenética mediana: 0.993


## 4. Comparación con R

Composición, nodo y representante exactos; correlaciones, intervalos, Rand ajustado, cofenética
y frecuencias de coclasificación con tolerancia $10^{-10}$.

In [5]:
for via, (M, d, dr) in VIAS.items():
    f_r = dr / 'R_hfc_clases3.csv'
    if not f_r.exists():
        print(f'{via}: aún no existen los R_hfc_* (ejecutar el Rmd)')
        continue
    rc = pd.read_csv(f_r)
    pc = comp[via]
    clave = lambda t: list(zip(t.particion, t.clase, t.nodo, t.codigo, t.representante.astype(str).str.upper()))
    assert clave(rc) == clave(pc), via
    difs = {'r_factor': float(np.max(np.abs(rc.r_factor - pc.r_factor)))}
    rr = pd.read_csv(dr / 'R_hfc_correlaciones3.csv')
    assert list(zip(rr.version, rr.tipo, rr.clase_a, rr.clase_b)) == list(zip(corr[via].version, corr[via].tipo, corr[via].clase_a, corr[via].clase_b))
    difs['correlaciones'] = float(np.max(np.abs(rr[['r', 'p5', 'p95']].to_numpy() - corr[via][['r', 'p5', 'p95']].to_numpy())))
    rb = pd.read_csv(dr / 'R_hfc_bootstrap.csv')
    difs['bootstrap'] = float(np.max(np.abs(rb[['ari_3', 'ari_4', 'cofenetica']].to_numpy() - boot[via][0][['ari_3', 'ari_4', 'cofenetica']].to_numpy())))
    rco = pd.read_csv(dr / 'R_hfc_coclasificacion.csv')
    assert list(rco.codigo_a + rco.codigo_b) == list(boot[via][1].codigo_a + boot[via][1].codigo_b)
    difs['coclasificacion'] = float(np.max(np.abs(rco[['frec_3', 'frec_4']].to_numpy() - boot[via][1][['frec_3', 'frec_4']].to_numpy())))
    assert max(difs.values()) < 1e-10, (via, difs)
    print(f'{via}: coincide con R ·', {k: f'{v:.1e}' for k, v in difs.items()})

niveles: coincide con R · {'r_factor': '4.4e-16', 'correlaciones': '1.6e-15', 'bootstrap': '8.9e-16', 'coclasificacion': '0.0e+00'}


**Lectura.** Sin interpretación (regla 7): las clases, sus factores y su estabilidad se leen con el
Dr. Camiz (apunte 12, OE5). Las clases se comparan por composición.